<a href="https://colab.research.google.com/github/shobha-nosimpler/GenAI/blob/main/genAi_category_from_description.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Using gpt-3.5-turbo-16 to output laptop details from laptop description

# Use case 1 : Determine category of the laptop

# Here we reade laptop description file and classify based on description into different categories
- general
- business
- gamer
- programmer
- multimedia

input laptop_description
------------------

0	The Dell Inspiron is a versatile laptop that c...

1	The MSI GL65 is a high-performance laptop desi...

2	The HP EliteBook is a premium laptop designed ...

3	The Lenovo IdeaPad is a versatile laptop that ...

4	The ASUS ZenBook Pro is a high-end laptop that...

output
---------
laptop_description	Category

0	The Dell Inspiron is a versatile laptop that c...	general

1	The MSI GL65 is a high-performance laptop desi...	gamer

2	The HP EliteBook is a premium laptop designed ...	business

3	The Lenovo IdeaPad is a versatile laptop that ...	general

4	The ASUS ZenBook Pro is a high-end laptop that...	gamer




# Testing openai API key

In [90]:
!pip install openai

In [91]:
import openai

In [ ]:
# pass secret key for authentication
api_key = input("Enter your API key: ")
openai.api_key = api_key

In [93]:
# Simple user request to test
messages=[
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Who won IPL 2020"}
    ]

# Make a chat completion API call using the GPT-3.5-turbo model
chat_response = openai.chat.completions.create(
  model="gpt-3.5-turbo-16k",
  messages=messages
)

# Print chat response as needed
chat_response

ChatCompletion(id='chatcmpl-9hanZFLlBZXQw6SvZ48bUpQ2mp4HK', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Mumbai Indians won the IPL 2020.', role='assistant', function_call=None, tool_calls=None))], created=1720176585, model='gpt-3.5-turbo-16k-0613', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=10, prompt_tokens=23, total_tokens=33))

# Read the laptop descriptions csv file into a data frame

In [94]:
import pandas as pd
import numpy as np


In [95]:
# Read the CSV file into a pandas DataFrame
df = pd.read_csv('/content/laptop_descriptions.csv')

# Display the DataFrame
print(df.head())

                                  laptop_description
0  The Dell Inspiron is a versatile laptop that c...
1  The MSI GL65 is a high-performance laptop desi...
2  The HP EliteBook is a premium laptop designed ...
3  The Lenovo IdeaPad is a versatile laptop that ...
4  The ASUS ZenBook Pro is a high-end laptop that...


# Prompt clearly explaining the categories

In [49]:
mcq1_prompt = '''
From the description of a laptop (delimited by '###'), you have to identify what role does the laptop serve. Refer to the key value pairs of categories and category details below. Identify which of the following details does the product description fits best and assign that category to that latpop. \n
Categories:
[
    'general': 'For general purpose use such as light web browsing, editing documents etc.'
    'business': 'For business users, the focus is on portability, battery backup and general purpose use.'
    'gamer': 'For gamers, the focus is primarily on high-performance, high-end graphics requirement, efficient processor etc.'
    'programmer': 'For programmers, the focus is on performance, battery backup, high-end RAM etc.'
    'multimedia': 'For multimedia use cases, the requirements are a good quality/ high resolution display, wide screens, good audio and video quality, battery backup, efficiency etc.' # Write the prompt here
] \n
Laptop description: {description}
'''

In [50]:
# Best practice
import openai
from tenacity import retry, wait_random_exponential, stop_after_attempt

# Retry up to 6 times with exponential backoff, starting at 1 second and maxing out at 20 seconds delay
@retry(wait=wait_random_exponential(min=1, max=20), stop=stop_after_attempt(6))
def get_chat_response_mcq1(user_request):

  '''
  This function ONLY takes `user_request` as the input argument.
  As you can see, the System Prompt is given inside the function itself so we don't require to give it as an input argument
  '''

  MODEL = 'gpt-3.5-turbo-16k'# Define GPT model

  SYSTEM_MESSAGE = '''You are a shopping assistant. The user will give you laptop description and some categories and their details. You have to find out which of the categories does the laptop fit best according to description. Remember to only give one word output, the category name, from the list of categories only, which resembles most closely.''' # Default System Message

  try:

    messages = [

                {"role" : 'system', "content": SYSTEM_MESSAGE},
                {"role" : 'user', "content": user_request}
    ] # Define the list of messages

    response = openai.chat.completions.create(
        model = MODEL,
        messages = messages
    ) # Get the ChatCompletion Response from the GPT-3.5 model


    # Parse the response_content from the message
    response_content = response.choices[0].message.content

    return response_content

  # Raise exception
  except Exception as e:
    print(f"An error occured: {e}")
    return None

In [51]:
laptop_df = df.copy()
laptop_df.head()

,laptop_description
0,The Dell Inspiron is a versatile laptop that c...
1,The MSI GL65 is a high-performance laptop desi...
2,The HP EliteBook is a premium laptop designed ...
3,The Lenovo IdeaPad is a versatile laptop that ...
4,The ASUS ZenBook Pro is a high-end laptop that...


In [52]:
# creates a list of dictionaries from the dataframe
# the column name is the key, and row is the value
# each dictionary has the same key, which is the column name
#and value is the row value

laptop_dict = laptop_df.to_dict(orient='records')
type(laptop_dict)

list

In [53]:
len(laptop_dict)

20

In [12]:
laptop_dict

[{'laptop_description': 'The Dell Inspiron is a versatile laptop that combines powerful performance and affordability. It features an Intel Core i5 processor clocked at 2.4 GHz, ensuring smooth multitasking and efficient computing. With 8GB of RAM and an SSD, it offers quick data access and ample storage capacity. The laptop sports a vibrant 15.6" LCD display with a resolution of 1920x1080, delivering crisp visuals and immersive viewing experience. Weighing just 2.5 kg, it is highly portable, making it ideal for on-the-go usage. Additionally, it boasts an Intel UHD GPU for decent graphical performance and a backlit keyboard for enhanced typing convenience. With a one-year warranty and a battery life of up to 6 hours, the Dell Inspiron is a reliable companion for work or entertainment. All these features are packed at an affordable price of 35,000, making it an excellent choice for budget-conscious users.'},
 {'laptop_description': 'The MSI GL65 is a high-performance laptop designed for

In [54]:
# access the first description form the list of descriptions
laptop_dict[0]['laptop_description']

'The Dell Inspiron is a versatile laptop that combines powerful performance and affordability. It features an Intel Core i5 processor clocked at 2.4 GHz, ensuring smooth multitasking and efficient computing. With 8GB of RAM and an SSD, it offers quick data access and ample storage capacity. The laptop sports a vibrant 15.6" LCD display with a resolution of 1920x1080, delivering crisp visuals and immersive viewing experience. Weighing just 2.5 kg, it is highly portable, making it ideal for on-the-go usage. Additionally, it boasts an Intel UHD GPU for decent graphical performance and a backlit keyboard for enhanced typing convenience. With a one-year warranty and a battery life of up to 6 hours, the Dell Inspiron is a reliable companion for work or entertainment. All these features are packed at an affordable price of 35,000, making it an excellent choice for budget-conscious users.'

In [55]:
# append the description to the mcq1_promt which has categories defined
prompt = mcq1_prompt.format(description=laptop_dict[0]['laptop_description'])
prompt

'\nFrom the description of a laptop (delimited by \'###\'), you have to identify what role does the laptop serve. Refer to the key value pairs of categories and category details below. Identify which of the following details does the product description fits best and assign that category to that latpop. \n\nCategories:\n[\n    \'general\': \'For general purpose use such as light web browsing, editing documents etc.\'\n    \'business\': \'For business users, the focus is on portability, battery backup and general purpose use.\'\n    \'gamer\': \'For gamers, the focus is primarily on high-performance, high-end graphics requirement, efficient processor etc.\'\n    \'programmer\': \'For programmers, the focus is on performance, battery backup, high-end RAM etc.\'\n    \'multimedia\': \'For multimedia use cases, the requirements are a good quality/ high resolution display, wide screens, good audio and video quality, battery backup, efficiency etc.\' # Write the prompt here\n] \n\nLaptop des

In [56]:
# assign the laptop category to the column laptop_category
#laptop_df.at[0,'Category'] = get_chat_response_mcq1(prompt)
get_chat_response_mcq1(prompt)

'multimedia'

In [59]:
# Define function to tag each laptop description with predefined category

def tag_laptop_description(df_laptop_desc, mcq1_prompt):
  # create a list of dictionaries with each row as value, and column name as key
  laptop_dict = df_laptop_desc.to_dict(orient='records')

  # For each record, that as the description create a prompt to get the category

  # Get the laptop category for each description

  for i in range(len(laptop_dict)):
    prompt = mcq1_prompt.format(description=laptop_dict[i]['laptop_description'])
    #laptop_dict[i]['Category'] = get_chat_response_mcq1(prompt)
    # Assign the laptop category to the column laptop_category
    laptop_category = get_chat_response_mcq1(prompt)
    df_laptop_desc.at[i,'Category'] = laptop_category

  return df_laptop_desc


In [60]:
tag_df = tag_laptop_description(df, mcq1_prompt)

In [15]:
print(df.head())

                                  laptop_description
0  The Dell Inspiron is a versatile laptop that c...
1  The MSI GL65 is a high-performance laptop desi...
2  The HP EliteBook is a premium laptop designed ...
3  The Lenovo IdeaPad is a versatile laptop that ...
4  The ASUS ZenBook Pro is a high-end laptop that...


In [61]:
print(tag_df.head())

                                  laptop_description    Category
0  The Dell Inspiron is a versatile laptop that c...     general
1  The MSI GL65 is a high-performance laptop desi...       gamer
2  The HP EliteBook is a premium laptop designed ...    business
3  The Lenovo IdeaPad is a versatile laptop that ...  multimedia
4  The ASUS ZenBook Pro is a high-end laptop that...       gamer


# Usecase 2: Extracting laptop properties from laptop description

# Instead of single output, get a json output with defined structure

In [17]:
structure = '''{
    "Brand": ___ ,
    "Model Name": ___ ,
    "GPU processor": ___ ,
    "Display Resolution": ___ ,
    "Weight": ___ ,
    "Processor": ___ ,
    "Clock speed": ___ ,
    "Budget": ___
}'''

In [18]:
mcq2_prompt = '''
Laptop Decription: {description}
From the laptop decription above, you have to extract relevant values for the following dictionary items. The dictionary structure should be as follows: {str}
Try giving quantitative, absolute, or numerical outputs. Try not to give qualitative or adjective outputs. for example: If the processing speed of a laptop is 2.4GHz, then, in the "processing speed" key, give output as '2.4GHz' instead of 'very fast'.
Extract only one word values of these properties. Fill in the blanks for each product and output each product's dictionary in json format.
'''

In [19]:
def get_chat_response_mcq2(user_request):

  MODEL = 'gpt-3.5-turbo-16k'# Define GPT model

  SYSTEM_MESSAGE = 'You are a helpful shopping assitant.'# Default System Message

  try:
  # Define the list of messages
    messages = [
        {'role': 'system', 'content': SYSTEM_MESSAGE},
        {'role': 'user', 'content': user_request}
    ]

# Get the ChatCompletion Response from the GPT-3.5 model
    response = openai.chat.completions.create(
        model = MODEL,
        messages = messages,
        #response_format = {'type': 'json_object'}
    )

    response_content = response.choices[0].message.content
    # Parse the response_content from the message

    return response_content

  # Raise exception error
  except Exception as e:
    print(f"An error occurred: {e}")
    return None

In [20]:
laptop_df.head()

,laptop_description,Category
0,The Dell Inspiron is a versatile laptop that c...,multimedia
1,The MSI GL65 is a high-performance laptop desi...,gamer
2,The HP EliteBook is a premium laptop designed ...,business
3,The Lenovo IdeaPad is a versatile laptop that ...,multimedia
4,The ASUS ZenBook Pro is a high-end laptop that...,gamer


In [44]:
laptop_dict

[{'laptop_description': 'The Dell Inspiron is a versatile laptop that combines powerful performance and affordability. It features an Intel Core i5 processor clocked at 2.4 GHz, ensuring smooth multitasking and efficient computing. With 8GB of RAM and an SSD, it offers quick data access and ample storage capacity. The laptop sports a vibrant 15.6" LCD display with a resolution of 1920x1080, delivering crisp visuals and immersive viewing experience. Weighing just 2.5 kg, it is highly portable, making it ideal for on-the-go usage. Additionally, it boasts an Intel UHD GPU for decent graphical performance and a backlit keyboard for enhanced typing convenience. With a one-year warranty and a battery life of up to 6 hours, the Dell Inspiron is a reliable companion for work or entertainment. All these features are packed at an affordable price of 35,000, making it an excellent choice for budget-conscious users.'},
 {'laptop_description': 'The MSI GL65 is a high-performance laptop designed for

In [45]:
#mcq2_prompt has two placeholders : description and str
prompt = mcq2_prompt.format(description=laptop_dict[0]['laptop_description'], str=structure)
prompt

'\nLaptop Decription: The Dell Inspiron is a versatile laptop that combines powerful performance and affordability. It features an Intel Core i5 processor clocked at 2.4 GHz, ensuring smooth multitasking and efficient computing. With 8GB of RAM and an SSD, it offers quick data access and ample storage capacity. The laptop sports a vibrant 15.6" LCD display with a resolution of 1920x1080, delivering crisp visuals and immersive viewing experience. Weighing just 2.5 kg, it is highly portable, making it ideal for on-the-go usage. Additionally, it boasts an Intel UHD GPU for decent graphical performance and a backlit keyboard for enhanced typing convenience. With a one-year warranty and a battery life of up to 6 hours, the Dell Inspiron is a reliable companion for work or entertainment. All these features are packed at an affordable price of 35,000, making it an excellent choice for budget-conscious users.\nFrom the laptop decription above, you have to extract relevant values for the follow

In [47]:
values = get_chat_response_mcq2(prompt)

In [48]:
print(values)

{
    "Brand": "Dell",
    "Model Name": "Inspiron",
    "GPU processor": "UHD",
    "Display Resolution": "1920x1080",
    "Weight": "2.5 kg",
    "Processor": "Intel Core i5",
    "Clock speed": "2.4 GHz",
    "Budget": "35,000"
}


In [21]:
# Let's define a function to extract information in JSON format for each laptop description
def extract_information(df_laptop_desc, mcq2_prompt):

  laptop_dict = df_laptop_desc.to_dict(orient='records')

  # Creating an empty list to store the properties
  result = []

  # Get the relevant values for each of the property, key:
  for i in range(len(laptop_dict)):
    prompt = mcq2_prompt.format(description=laptop_dict[i]['laptop_description'], str=structure)
    values = get_chat_response_mcq2(prompt)
    result.append(values)
    # We will print each dictionary property for the current laptop descriptioin
    print(result[i])
  # But in the function output, we are returning the whole list as a whole
  return result

  # Your solution can differ, but your end goal is to output the porperties for all products in one single list

In [22]:
values = extract_information(df, mcq2_prompt)

{
    "Brand": "Dell",
    "Model Name": "Inspiron",
    "GPU processor": "Intel",
    "Display Resolution": "1920x1080",
    "Weight": "2.5 kg",
    "Processor": "Intel Core i5",
    "Clock speed": "2.4 GHz",
    "Budget": "35000"
}
{
    "Brand": "MSI",
    "Model Name": "GL65",
    "GPU processor": "NVIDIA",
    "Display Resolution": "1920x1080",
    "Weight": "2.3 kg",
    "Processor": "Intel",
    "Clock speed": "2.6 GHz",
    "Budget": "55,000"
}

{
    "Brand": "Dell",
    "Model Name": "XPS 15",
    "GPU processor": "NVIDIA",
    "Display Resolution": "3840x2160",
    "Weight": "4.5 lbs",
    "Processor": "Intel",
    "Clock speed": "2.4 GHz",
    "Budget": "90,000"
}

{
    "Brand": "HP",
    "Model Name": "Omen 15",
    "GPU processor": "NVIDIA",
    "Display Resolution": "1920x1080",
    "Weight": "5.2 lbs",
    "Processor": "Intel",
    "Clock speed": "2.3 GHz",
    "Budget": "70,000"
}
{
    "Brand": "HP",
    "Model Name": "EliteBook",
    "GPU processor": "Intel UHD",
  

In [24]:
print(type(values))

<class 'list'>


In [25]:
print(values[0])

{
    "Brand": "Dell",
    "Model Name": "Inspiron",
    "GPU processor": "Intel",
    "Display Resolution": "1920x1080",
    "Weight": "2.5 kg",
    "Processor": "Intel Core i5",
    "Clock speed": "2.4 GHz",
    "Budget": "35000"
}


In [27]:
print(values[0])

{
    "Brand": "Dell",
    "Model Name": "Inspiron",
    "GPU processor": "Intel",
    "Display Resolution": "1920x1080",
    "Weight": "2.5 kg",
    "Processor": "Intel Core i5",
    "Clock speed": "2.4 GHz",
    "Budget": "35000"
}


#Usecase 3: Combinig usecase 1 and usecase 2
Assign laptop category from description and include it in the laptop properties

# Create a structure to include category

Combine mcq1_prompt and mcq2_promt to get all details in single structure

In [96]:
laptop_properties = '''{
  "Category": ___ ,
  "Brand": ___ ,
  "Model Name": ___ ,
  "GPU processor": ___ ,
  "Display Resolution": ___ ,
  "Weight": ___ ,"Processor": ___ ,
  "Clock speed": ___ ,
  "Budget": ___}
  '''

In [97]:
mcq3_prompt = '''
Laptop Decription: {description}
From the laptop decription above, you have to extract relevant values for the laptop properties for the following dictionary items. The dictionary structure should be as follows: {properties}
For the first property 'Category', you have to identify what role does the laptop serve. Refer to the key value pairs of categories and category details below. Identify which of the following details does the product description fits best and assign that category to that latpop. \n
Categories:
[
    'general': 'For general purpose use such as light web browsing, editing documents etc.'
    'business': 'For business users, the focus is on portability, battery backup and general purpose use.'
    'gamer': 'For gamers, the focus is primarily on high-performance, high-end graphics requirement, efficient processor etc.'
    'programmer': 'For programmers, the focus is on performance, battery backup, high-end RAM etc.'
    'multimedia': 'For multimedia use cases, the requirements are a good quality/ high resolution display, wide screens, good audio and video quality, battery backup, efficiency etc.' # Write the prompt here
] \n
For the reamining properties try giving quantitative, absolute, or numerical outputs. Try not to give qualitative or adjective outputs. For example: If the processing speed of a laptop is 2.4GHz, then, in the "processing speed" key, give output as '2.4GHz' instead of 'very fast'.
Extract only one word values of these properties. Fill in the blanks for each product and output each product's dictionary in json format. The output dictionary should have no special characters, it should contain key and values paris containing only characters and/or numbers.
'''

# get chat response

In [98]:
# Best practice
import openai
from tenacity import retry, wait_random_exponential, stop_after_attempt

# Retry up to 6 times with exponential backoff, starting at 1 second and maxing out at 20 seconds delay
@retry(wait=wait_random_exponential(min=1, max=20), stop=stop_after_attempt(6))
def get_chat_response_mcq3(user_request):

  MODEL = 'gpt-3.5-turbo-16k'# Define GPT model

  SYSTEM_MESSAGE = 'You are a helpful shopping assitant.'# Default System Message

  try:
  # Define the list of messages
    messages = [
        {'role': 'system', 'content': SYSTEM_MESSAGE},
        {'role': 'user', 'content': user_request}
    ]

# Get the ChatCompletion Response from the GPT-3.5 model
    response = openai.chat.completions.create(
        model = MODEL,
        messages = messages,
        #response_format = {'type': 'json_object'}
    )

    response_content = response.choices[0].message.content
    # Parse the response_content from the message

    return response_content

  # Raise exception error
  except Exception as e:
    print(f"An error occurred: {e}")
    return None

## Testing for one laptop description

In [99]:
laptop_df = df.copy()
laptop_df.head()

,laptop_description
0,The Dell Inspiron is a versatile laptop that c...
1,The MSI GL65 is a high-performance laptop desi...
2,The HP EliteBook is a premium laptop designed ...
3,The Lenovo IdeaPad is a versatile laptop that ...
4,The ASUS ZenBook Pro is a high-end laptop that...


In [100]:
laptop_dict = laptop_df.to_dict(orient='records')
type(laptop_dict)

list

In [101]:
len(laptop_dict)

20

In [102]:
# access the first description form the list of descriptions
laptop_dict[7]['laptop_description']

'The Lenovo ThinkPad is a powerful laptop designed for professional users. It is equipped with a Ryzen 7 processor from AMD clocked at 3.0 GHz, providing strong processing capabilities for demanding tasks. With 16GB of RAM and an SSD, it offers smooth multitasking and fast storage access. The laptop features a 14" IPS display with a resolution of 2560x1440, delivering sharp visuals and accurate colors. It also comes with an NVIDIA GTX graphics card for enhanced graphical performance. Weighing just 1.6 kg, it is lightweight and highly portable. The laptop features a backlit keyboard for comfortable typing in low-light environments. With a three-year warranty and a battery life of up to 6 hours, the Lenovo ThinkPad offers reliability and durability. Priced at 60,000, it is an excellent choice for professionals seeking powerful performance and a versatile display.'

In [103]:
# append the description to the mcq1_promt which has categories defined
prompt = mcq3_prompt.format(description=laptop_dict[7]['laptop_description'], properties=laptop_properties)
prompt

'\nLaptop Decription: The Lenovo ThinkPad is a powerful laptop designed for professional users. It is equipped with a Ryzen 7 processor from AMD clocked at 3.0 GHz, providing strong processing capabilities for demanding tasks. With 16GB of RAM and an SSD, it offers smooth multitasking and fast storage access. The laptop features a 14" IPS display with a resolution of 2560x1440, delivering sharp visuals and accurate colors. It also comes with an NVIDIA GTX graphics card for enhanced graphical performance. Weighing just 1.6 kg, it is lightweight and highly portable. The laptop features a backlit keyboard for comfortable typing in low-light environments. With a three-year warranty and a battery life of up to 6 hours, the Lenovo ThinkPad offers reliability and durability. Priced at 60,000, it is an excellent choice for professionals seeking powerful performance and a versatile display.\nFrom the laptop decription above, you have to extract relevant values for the laptop properties for the 

In [104]:
values = get_chat_response_mcq3(prompt)

In [105]:
print(values)

{
  "Category": "business",
  "Brand": "Lenovo",
  "Model Name": "ThinkPad",
  "GPU processor": "NVIDIA",
  "Display Resolution": "2560x1440",
  "Weight": "1.6kg",
  "Processor": "Ryzen",
  "Clock speed": "3.0GHz",
  "Budget": "60000"
}


## Return the properties for all the laptop descriptions

We have tested the response for the first laptop property, let's call the gpt to return for each laptop description in a loop.

In [106]:
def extract_information(df_laptop_description,mcq3_prompt):

  #laptop_df = df.copy()
  laptop_dict = df_laptop_description.to_dict(orient ='records')

  # Creating an empty list to store the properties
  result = []

  # Get the relevant values for each of the property:
  for i in range(len(laptop_dict)):
    prompt = mcq3_prompt.format(description=laptop_dict[i]['laptop_description'], properties=laptop_properties)
    values = get_chat_response_mcq3(prompt)
    result.append(values)
    # Print each of the dictionary of properties
    print(f"For description # {i} : {result[i]}")
    #print(result[i])
  # But in the function output, we are returning the whole list as a whole.
  return result


In [107]:
all_laptop_properties = extract_information(df, mcq3_prompt)

For description # 0 : {
  "Category": "general",
  "Brand": "Dell",
  "Model Name": "Inspiron",
  "GPU processor": "Intel",
  "Display Resolution": "1920x1080",
  "Weight": "2.5",
  "Processor": "Intel Core i5",
  "Clock speed": "2.4 GHz",
  "Budget": "35000"
}
For description # 1 : {
  "Category": "gamer",
  "Brand": "MSI",
  "Model Name": "GL65",
  "GPU processor": "NVIDIA",
  "Display Resolution": "1920x1080",
  "Weight": "2.3",
  "Processor": "Intel",
  "Clock speed": "2.6",
  "Budget": "55000"
}
For description # 2 : {
  "Category": "business",
  "Brand": "HP",
  "Model Name": "EliteBook",
  "GPU processor": "Intel UHD",
  "Display Resolution": "1920x1080",
  "Weight": "1.5",
  "Processor": "Intel",
  "Clock speed": "2.8",
  "Budget": "90000"
}
For description # 3 : {
  "Category": "general",
  "Brand": "Lenovo",
  "Model Name": "IdeaPad",
  "GPU processor": "Intel UHD",
  "Display Resolution": "1366x768",
  "Weight": "2.2kg",
  "Processor": "Intel Core i3",
  "Clock speed": "2.1G

# To Do
Convert the string of dictionaries into a JSON format and create csv file with keys as header

In [108]:
print(all_laptop_properties)

['{\n  "Category": "general",\n  "Brand": "Dell",\n  "Model Name": "Inspiron",\n  "GPU processor": "Intel",\n  "Display Resolution": "1920x1080",\n  "Weight": "2.5",\n  "Processor": "Intel Core i5",\n  "Clock speed": "2.4 GHz",\n  "Budget": "35000"\n}', '{\n  "Category": "gamer",\n  "Brand": "MSI",\n  "Model Name": "GL65",\n  "GPU processor": "NVIDIA",\n  "Display Resolution": "1920x1080",\n  "Weight": "2.3",\n  "Processor": "Intel",\n  "Clock speed": "2.6",\n  "Budget": "55000"\n}', '{\n  "Category": "business",\n  "Brand": "HP",\n  "Model Name": "EliteBook",\n  "GPU processor": "Intel UHD",\n  "Display Resolution": "1920x1080",\n  "Weight": "1.5",\n  "Processor": "Intel",\n  "Clock speed": "2.8",\n  "Budget": "90000"\n}', '{\n  "Category": "general",\n  "Brand": "Lenovo",\n  "Model Name": "IdeaPad",\n  "GPU processor": "Intel UHD",\n  "Display Resolution": "1366x768",\n  "Weight": "2.2kg",\n  "Processor": "Intel Core i3",\n  "Clock speed": "2.1GHz",\n  "Budget": "25000"\n}', '{\n  "C

In [83]:
type(all_laptop_properties)

list

In [86]:
type(all_laptop_properties[0])

str

In [109]:
all_laptop_properties[0]

'{\n  "Category": "general",\n  "Brand": "Dell",\n  "Model Name": "Inspiron",\n  "GPU processor": "Intel",\n  "Display Resolution": "1920x1080",\n  "Weight": "2.5",\n  "Processor": "Intel Core i5",\n  "Clock speed": "2.4 GHz",\n  "Budget": "35000"\n}'

In [110]:
all_laptop_properties[0].replace("\\n", "").replace("\\", "").replace("  ", "").replace("\n", "")

'{"Category": "general","Brand": "Dell","Model Name": "Inspiron","GPU processor": "Intel","Display Resolution": "1920x1080","Weight": "2.5","Processor": "Intel Core i5","Clock speed": "2.4 GHz","Budget": "35000"}'

In [89]:
str1 = all_laptop_properties[0].replace("\\n", "").replace("\\", "").replace("  ", "").replace("\n", "")
print(str1)
print(type(str1))

{"Category": "general","Brand": "Dell","Model Name": "Inspiron","GPU processor": "Intel","Display Resolution": "1920x1080","Weight": "2.5kg","Processor": "Intel","Clock speed": "2.4GHz","Budget": "35000"}
<class 'str'>


In [47]:
len(all_laptop_properties)

20

In [60]:
all_laptop_properties[0]

'{\n    "Category": "general",\n    "Brand": "Dell",\n    "Model Name": "Inspiron",\n    "GPU processor": "Intel",\n    "Display Resolution": "1920x1080",\n    "Weight": "2.5kg",\n    "Processor": "Intel",\n    "Clock speed": "2.4GHz",\n    "Budget": "35000"\n}'

In [61]:
all_laptop_properties

['{\n    "Category": "general",\n    "Brand": "Dell",\n    "Model Name": "Inspiron",\n    "GPU processor": "Intel",\n    "Display Resolution": "1920x1080",\n    "Weight": "2.5kg",\n    "Processor": "Intel",\n    "Clock speed": "2.4GHz",\n    "Budget": "35000"\n}',
 '{\n    "Category": "gamer",\n    "Brand": "MSI",\n    "Model Name": "GL65",\n    "GPU processor": "NVIDIA",\n    "Display Resolution": "1920x1080",\n    "Weight": "2.3kg",\n    "Processor": "Intel",\n    "Clock speed": "2.6GHz",\n    "Budget": "55,000"\n}',
 '{\n    "Category": "business",\n    "Brand": "HP",\n    "Model Name": "EliteBook",\n    "GPU processor": "Intel",\n    "Display Resolution": "1920x1080",\n    "Weight": "1.5kg",\n    "Processor": "Intel Core i7",\n    "Clock speed": "2.8GHz",\n    "Budget": "90,000"\n}',
 '{\n  "Category": "general",\n  "Brand": "Lenovo",\n  "Model Name": "IdeaPad",\n  "GPU processor": "Intel UHD",\n  "Display Resolution": "1366x768",\n  "Weight": "2.2kg",\n  "Processor": "Intel Core i

In [62]:
import json

In [79]:
data = all_laptop_properties

In [80]:
# convert list of dictionaries to JSON
json_data = json.dumps(data)
print(json_data)

["{\n    \"Category\": \"general\",\n    \"Brand\": \"Dell\",\n    \"Model Name\": \"Inspiron\",\n    \"GPU processor\": \"Intel\",\n    \"Display Resolution\": \"1920x1080\",\n    \"Weight\": \"2.5kg\",\n    \"Processor\": \"Intel\",\n    \"Clock speed\": \"2.4GHz\",\n    \"Budget\": \"35000\"\n}", "{\n    \"Category\": \"gamer\",\n    \"Brand\": \"MSI\",\n    \"Model Name\": \"GL65\",\n    \"GPU processor\": \"NVIDIA\",\n    \"Display Resolution\": \"1920x1080\",\n    \"Weight\": \"2.3kg\",\n    \"Processor\": \"Intel\",\n    \"Clock speed\": \"2.6GHz\",\n    \"Budget\": \"55,000\"\n}", "{\n    \"Category\": \"business\",\n    \"Brand\": \"HP\",\n    \"Model Name\": \"EliteBook\",\n    \"GPU processor\": \"Intel\",\n    \"Display Resolution\": \"1920x1080\",\n    \"Weight\": \"1.5kg\",\n    \"Processor\": \"Intel Core i7\",\n    \"Clock speed\": \"2.8GHz\",\n    \"Budget\": \"90,000\"\n}", "{\n  \"Category\": \"general\",\n  \"Brand\": \"Lenovo\",\n  \"Model Name\": \"IdeaPad\",\n  \

In [82]:
print(type(json_data[0]))

<class 'str'>


In [65]:
import pandas as pd
try:
  df_data = pd.DataFrame(json_data)
  print(df_data)
except Exception as e:
  print(f"An error occurred: {e}")

An error occurred: DataFrame constructor not properly called!


In [66]:
import json
import re

In [67]:
# Function to remove special characters
def remove_special_characters(value):
    # Define pattern for special characters
    pattern = r'[\n\\]'
    # Remove special characters
    return re.sub(pattern, '', value)

In [73]:
data1 = input_string.replace("\\n", "").replace("\\", "").replace("  ", "").replace("\n", "")

In [74]:
data1

'{"Category": "general","Brand": "Dell","Model Name": "Inspiron","GPU processor": "Intel","Display Resolution": "1920x1080","Weight": "2.5kg","Processor": "Intel","Clock speed": "2.4GHz","Budget": "35000"}'

In [76]:
# Clean dictionary values
data = {"Category": "general","Brand": "Dell","Model Name": "Inspiron","GPU processor": "Intel","Display Resolution": "1920x1080","Weight": "2.5kg","Processor": "Intel","Clock speed": "2.4GHz","Budget": "35000"}
cleaned_data = {key: remove_special_characters(value) for key, value in data.items()}


In [77]:
cleaned_data

{'Category': 'general',
 'Brand': 'Dell',
 'Model Name': 'Inspiron',
 'GPU processor': 'Intel',
 'Display Resolution': '1920x1080',
 'Weight': '2.5kg',
 'Processor': 'Intel',
 'Clock speed': '2.4GHz',
 'Budget': '35000'}

In [ ]:
input_string = '"{\n    \"Category\": \"general\",\n    \"Brand\": \"Dell\",\n    \"Model Name\": \"Inspiron\",\n    \"GPU processor\": \"Intel\",\n    \"Display Resolution\": \"1920x1080\",\n    \"Weight\": \"2.5kg\",\n    \"Processor\": \"Intel\",\n    \"Clock speed\": \"2.4GHz\",\n    \"Budget\": \"35000\"\n}"'


In [68]:
input_string = """{
    \"Category\": \"general\",
    \"Brand\": \"Dell\",
    \"Model Name\": \"Inspiron\",
    \"GPU processor\": \"Intel\",
    \"Display Resolution\": \"1920x1080\",
    \"Weight\": \"2.5kg\",
    \"Processor\": \"Intel\",
    \"Clock speed\": \"2.4GHz\",
    \"Budget\": \"35000\"
}"""

In [71]:
cleaned_string = input_string.replace("\\n", "").replace("\\", "").replace("  ", "").replace("\n", "")

In [72]:
cleaned_string

'{"Category": "general","Brand": "Dell","Model Name": "Inspiron","GPU processor": "Intel","Display Resolution": "1920x1080","Weight": "2.5kg","Processor": "Intel","Clock speed": "2.4GHz","Budget": "35000"}'